# PyTorch torch.nn 完整深度教程
参考文档：https://www.runoob.com/pytorch/pytorch-torch-nn-ref.html

## 学习目标
1. 深度理解 `nn.Module` 底层机制、参数注册、三大模块容器核心差异
2. 掌握 Sequential / ModuleList / ModuleDict 选型、嵌套、工程实战用法
3. 精通 torch.nn 全部常用网络层：线性、激活、卷积、池化、归一化、Dropout
4. 熟练使用各类损失函数、优化器，理解梯度更新完整流程
5. 独立搭建完整训练+评估流水线，实现分类模型实战
6. 规避 Module、容器、网络训练高频踩坑点

In [ ]:
# ====================== 全局统一依赖导入 ======================
# 全程仅此处统一导入所有库，后续代码Cell不再重复import，减少冗余
# 1. pytorch核心基础库，张量计算核心
import torch
# 2. 神经网络模块库，所有层/模型/损失函数均在此包下
import torch.nn as nn
# 3. 优化器库，负责梯度反向传播后更新权重
import torch.optim as optim
# 4. 无状态神经网络函数，无需实例化层直接调用计算
import torch.nn.functional as F
# 5. 有序字典，用于命名nn.Sequential网络层
from collections import OrderedDict
# 6. 数值计算辅助库，生成基础数组数据
import numpy as np

# ====================== 自动设备选择配置 ======================
# 判断环境是否存在可用NVIDIA GPU，存在则使用cuda加速，否则使用CPU
# torch.device统一封装设备标识，后续模型/张量统一to(device)迁移
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 打印当前运行设备，方便用户确认硬件加速状态
print(f"当前运行设备: {device}")

# 一、torch.nn 核心基类 nn.Module 完整讲解
## 核心概念梳理
`torch.nn.Module` 是所有网络层、模型、容器的父类，自定义网络必须继承；模型仅需实现两部分：
1. `__init__`：初始化所有网络层/子模块，绑定self才会注册参数
2. `forward`：定义张量前向传播数据流，model(x)自动调用，禁止手动forward(x)

批量层管理提供三类容器，全部继承自`nn.Module`：`Sequential` / `ModuleList` / `ModuleDict`

### 为什么不能用原生 list / dict 存储网络层？
普通Python容器不会递归注册weight/bias参数，无法参与梯度训练、设备迁移、权重保存；三大专用容器自动完成参数注册，配套nn全套能力。

### 容器核心分水岭：是否自带forward自动数据流
- `nn.Sequential`：内置forward，串行自动传递张量，可独立推理
- `ModuleList/ModuleDict`：仅做存储，无内置前向逻辑，forward内必须手动调度层

### ModuleList vs ModuleDict 基础区分
1. ModuleList：数字下标、有序列表 → 重复堆叠块、多任务同步联合训练
2. ModuleDict：字符串key键值对 → 多分支任务、推理按需单支路运行节省算力

## 1.1 nn.Module 全局通用底层能力
所有网络、容器共享统一接口：
- 参数遍历：`parameters()`、`named_parameters()` 查看可训练权重
- 设备迁移：`.to(device)` / `.cuda()` / `.cpu()` 一键整体迁移
- 权重持久化：`.state_dict()` 保存、加载模型权重
- 模式切换：`.train()`（训练，Dropout/BatchNorm生效） / `.eval()`（推理，关闭正则）
- 自动接入autograd反向传播

### 层级分工边界
1. 自定义`nn.Module`：顶层模型外壳，承载完整前向逻辑，程序推理入口
2. 三类容器：仅作为模型内部工具批量收纳层，一般不单独作为顶层网络
3. 核心判断标准：容器是否内置层间张量自动传递逻辑

## 1.2 自定义模型强制实现两个核心方法
1. `__init__(self)`：初始化层，**必须赋值self.xxx**，否则参数不注册、无法训练
2. `forward(self, x)`：数据流定义；调用 `model(x)` 底层自动执行，禁止手动 `model.forward(x)`
- Sequential自带串行计算逻辑，无需手写数据流
- ModuleList/ModuleDict无执行逻辑，forward内部必须手动循环/索引调度张量

In [ ]:
# ====================== 最简自定义全连接模型 ======================
# 双层全连接分类网络，演示nn.Module标准模板
class Net(nn.Module):
    # 构造函数：初始化网络所有子层，入参：输入维度、隐藏层神经元、输出类别数
    def __init__(self, in_dim, hidden_dim, out_dim):
        # 强制调用父类nn.Module构造方法，不写则参数完全无法注册，模型失效
        super().__init__()
        # 将网络层绑定self实例属性，Module会自动扫描并注册weight/bias参数
        # 第一层全连接：输入维度映射到隐藏层维度
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        # 激活函数ReLU，引入非线性，否则多层等价单层线性变换
        self.relu = nn.ReLU()
        # 第二层全连接：隐藏层映射到输出类别维度
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    # 前向传播函数：定义张量数据流转规则，x为输入特征张量
    def forward(self, x):
        # 第一步：输入经过第一层全连接线性变换
        x = self.fc1(x)
        # 第二步：线性结果经过ReLU激活，过滤负值
        x = self.relu(x)
        # 第三步：激活后特征送入输出层，返回原始logits（未softmax）
        return self.fc2(x)

# ====================== 模型实例化与推理测试 ======================
# 创建模型实例：输入10维特征，隐藏层32个神经元，2分类任务输出
model = Net(in_dim=10, hidden_dim=32, out_dim=2).to(device)
# 生成测试输入张量：batch_size=5（5条样本），每条样本10维特征，自动迁移至对应设备
test_x = torch.randn(5, 10).to(device)
# 模型推理：直接调用model(test_x)，底层自动执行forward函数，禁止手动model.forward(test_x)
pred = model(test_x)
# 打印输出张量形状：[batch_size, out_dim] = [5, 2]
print("基础模型输出shape:", pred.shape)

# ====================== 遍历查看模型全部可训练参数 ======================
print("\n模型参数列表：")
# named_parameters() 返回(参数名称, 参数张量)迭代器，便于查看每层权重维度与梯度开关
for name, param in model.named_parameters():
    # name：参数归属层+权重/偏置标识；param.shape：参数矩阵维度；requires_grad：是否开启梯度更新
    print(f"{name}: shape={param.shape}, requires_grad={param.requires_grad}")

# 二、三大模块容器深度详解
## 2.1 容器统一底层优势
1. 自动递归注册内部所有子模块，原生list/dict无此功能
2. 完整继承nn.Module全部能力：设备迁移、权重保存、train/eval模式切换
3. 支持循环批量创建网络层，消除重复冗余代码

## ModuleList / ModuleDict 基础对比表
| 对比项 | nn.ModuleList | nn.ModuleDict |
|--------|--------------|--------------|
| 存储结构 | 有序列表 | 字符串键字典 |
| 索引方式 | 数字下标 layers[0] | 语义key heads["cls"] |
| 典型场景 | 重复堆叠、同步多任务联合训练 | 多分支、推理按需单支路执行 |
| 添加层 | .append() | dict["key"] = 网络层 |

## 2.2 nn.Sequential：串行流水线容器
内置重写forward，严格按传入顺序自动传递张量，无需手动写数据流，可独立推理。
限制：仅支持纯串行结构，无法实现分支、残差、条件跳转逻辑。
适用场景：分类输出头、简单CNN主干、无分支线性堆叠网络

In [ ]:
# ====================== Sequential写法1：无命名层 ======================
# 直接传入层实例，自动按0/1/2数字下标索引，适合极简串行网络
seq_plain = nn.Sequential(
    nn.Linear(10, 32),  # 第一层线性变换
    nn.ReLU(),          # 非线性激活
    nn.Linear(32, 2)    # 输出层
).to(device) # 整体容器迁移至GPU/CPU
# 构造测试输入：5条10维样本
test_input = torch.randn(5, 10).to(device)
# 容器自带forward，可直接传入张量推理
out_plain = seq_plain(test_input)
# 输出形状 [5,2]
print("无命名Sequential输出shape:", out_plain.shape)

In [ ]:
# ====================== Sequential写法2：OrderedDict命名层 ======================
# 使用有序字典为每层自定义字符串名称，支持seq_named.xxx直接访问指定层，方便微调、权重可视化
seq_named = nn.Sequential(OrderedDict([
    ("fc1", nn.Linear(10, 32)), # 命名第一层fc1
    ("act", nn.ReLU()),         # 命名激活层act
    ("fc2", nn.Linear(32, 2))   # 命名输出层fc2
])).to(device)
# 通过自定义名称直接获取指定网络层对象，用于单独修改权重、冻结层等操作
print("命名层fc1:", seq_named.fc1)

## 2.3 nn.ModuleList：网络层有序列表
仅提供列表存储能力，无forward实现，**不能单独传入张量推理**；必须封装进自定义模型，forward内手动循环传递张量。
适用场景：Transformer编码器堆叠、重复卷积块、多任务同步联合训练

In [ ]:
# ====================== ModuleList循环堆叠多层示例 ======================
# 误区：ModuleList仅存储层，无内置前向计算逻辑，禁止直接调用layers(x)，会抛出报错
class MultiStackNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 初始化空ModuleList容器，用于批量存放重复网络层
        self.layers = nn.ModuleList()
        # 循环5次，批量创建5个输入输出均为16维的全连接层，存入容器
        for _ in range(5):
            self.layers.append(nn.Linear(16, 16))

    def forward(self, x):
        # 必须手动循环遍历容器内所有层，逐层传递张量，框架不会自动执行
        for layer in self.layers:
            x = layer(x)
        # 返回经过全部堆叠层后的特征张量
        return x

# 实例化堆叠网络模型并迁移设备
stack_model = MultiStackNet().to(device)
# 测试输入：batch=3，16维特征
stack_in = torch.randn(3, 16).to(device)
# 完整前向推理
stack_out = stack_model(stack_in)
# 输出shape不变 [3,16]
print("ModuleList堆叠网络输出shape:", stack_out.shape)

## 2.4 nn.ModuleDict：多分支字典容器
仅提供键值存储，无forward实现，无法独立推理；__init__一次性创建全部分支，forward通过key动态选择支路计算。
区分：仅运行时选择分支属于动态路由，不是动态构建网络；动态新建层会导致权重无法保存/加载。
适用场景：多任务分类/分割/回归多头，推理仅运行目标分支，大幅节省算力

In [ ]:
# ====================== ModuleDict多任务双分支基础示例 ======================
# 共享主干特征，分出分类、分割两个任务输出头，推理按需选择分支
class MultiTaskNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ModuleDict字典容器，key为任务名称字符串，value为对应任务输出头网络层
        self.heads = nn.ModuleDict({
            "cls": nn.Linear(64, 10),   # cls分支：10分类任务，输入共享64维特征，输出10类别
            "seg": nn.Linear(64, 3)     # seg分支：分割任务，输出3通道特征图
        })

    # forward入参feat为共享主干输出特征，task指定需要执行的任务分支key，默认分类
    def forward(self, feat, task="cls"):
        # 根据传入任务字符串key，精准选取对应分支计算，跳过其余分支节省计算资源
        return self.heads[task](feat)

# 实例化多任务模型并迁移设备
task_model = MultiTaskNet().to(device)
# 模拟主干网络输出共享特征：batch=2，64维
shared_feat = torch.randn(2, 64).to(device)
# 执行分类分支推理
cls_res = task_model(shared_feat, task="cls")
# 执行分割分支推理
seg_res = task_model(shared_feat, task="seg")
# 打印两个分支输出维度
print("分类头输出shape:", cls_res.shape)
print("分割头输出shape:", seg_res.shape)

## 2.5 ModuleList / ModuleDict 同场景完整对比实例
场景：3分支网络（分类/分割/回归），演示批量全分支联合训练、单分支推理两种模式
误区纠正：
1. ModuleList不是必须跑完所有分支，支持下标单独调取；但数字索引可读性差，新增分支会打乱下标序号
2. ModuleDict不是只能单分支，可遍历.items()计算全部任务；语义key可读性强，推理可只跑目标支路

选型规则：
- 有序重复堆叠、同步训练全部分支 → ModuleList
- 多任务按需推理、分支语义区分 → ModuleDict
- 仅同步训练全分支场景可临时互换，其余场景不可替代

In [ ]:
# ====================== ModuleList 三分支完整测试代码 ======================
# 功能1：前向默认遍历所有分支，用于多任务联合训练；功能2：指定下标单独运行单分支推理
class ListBranchNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ModuleList有序存储3个任务输出头，下标固定对应任务：0=分类，1=分割，2=回归
        self.branchs = nn.ModuleList([
            nn.Linear(64, 10),  # 下标0：分类分支，输出10个类别logits
            nn.Linear(64, 3),   # 下标1：分割分支，输出3通道预测
            nn.Linear(64, 1)    # 下标2：回归分支，单值连续预测
        ])

    def forward(self, feat):
        """默认前向传播：遍历全部分支，一次性计算所有任务输出，多任务联合训练场景使用"""
        output_list = [] # 列表存储各分支输出张量
        # 循环遍历容器内全部分支层
        for branch_layer in self.branchs:
            branch_out = branch_layer(feat) # 单分支前向计算
            output_list.append(branch_out)  # 保存输出结果
        return output_list # 返回包含3个任务输出的列表

    def get_single_branch(self, feat, branch_idx):
        """自定义方法：指定数字下标，仅执行单一分支，推理时减少算力消耗"""
        # 通过数字下标选取指定分支层，单独计算输出
        return self.branchs[branch_idx](feat)

# 实例化模型并迁移至GPU/CPU
list_net = ListBranchNet().to(device)
# 模拟共享主干特征：batch=4，64维
test_feature = torch.randn(4, 64).to(device)

# ========== 测试1：批量执行全部3个分支（多任务联合训练场景） ==========
all_branch_outputs = list_net(test_feature)
print("===== ModuleList 批量全分支输出 =====")
print(f"分类(下标0) shape: {all_branch_outputs[0].shape}")
print(f"分割(下标1) shape: {all_branch_outputs[1].shape}")
print(f"回归(下标2) shape: {all_branch_outputs[2].shape}")

# ========== 测试2：仅单独执行分类单分支推理（下标0） ==========
single_cls_out = list_net.get_single_branch(test_feature, branch_idx=0)
print("\n===== ModuleList 单分支推理(仅分类) =====")
print(f"单分类输出shape: {single_cls_out.shape}")

In [ ]:
# ====================== ModuleDict 三分支完整测试代码 ======================
# 功能1：传入任务key仅执行单分支（推理常用，节省算力）；功能2：遍历全部任务联合训练
class DictBranchNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ModuleDict使用语义字符串key区分任务，可读性远优于数字下标，新增分支不影响旧索引
        self.branchs = nn.ModuleDict({
            "cls": nn.Linear(64, 10),  # cls：分类任务标识
            "seg": nn.Linear(64, 3),   # seg：分割任务标识
            "reg": nn.Linear(64, 1)    # reg：回归任务标识
        })

    def forward(self, feat, task_name):
        """默认前向：传入任务名字符串，仅执行目标分支，推理阶段优先使用"""
        # 根据任务key精准调取对应输出头，不计算其余分支
        return self.branchs[task_name](feat)

    def get_all_task_outputs(self, feat):
        """自定义方法：遍历字典全部任务分支，一次性计算所有输出，用于多任务联合训练"""
        output_dict = {} # 字典存储{任务名:输出张量}
        # 遍历ModuleDict内所有key与层实例
        for task_key, branch_layer in self.branchs.items():
            output_dict[task_key] = branch_layer(feat)
        return output_dict

# 实例化模型并迁移设备
dict_net = DictBranchNet().to(device)
test_feature = torch.randn(4, 64).to(device)

# ========== 测试1：按需单独执行分割分支（推理场景） ==========
seg_only_out = dict_net(test_feature, task_name="seg")
print("===== ModuleDict 按需单分支推理(分割) =====")
print(f"分割输出shape: {seg_only_out.shape}")

# ========== 测试2：一次性计算全部三类任务（联合训练场景） ==========
all_task_res = dict_net.get_all_task_outputs(test_feature)
print("\n===== ModuleDict 批量所有任务输出 =====")
for task, tensor_out in all_task_res.items():
    print(f"{task} 输出shape: {tensor_out.shape}")

## 2.6 三类容器总对比表 + 混合嵌套工程完整示例
| 容器 | 内置forward | 独立推理 | 索引方式 | 核心适用场景 |
|------|-------------|----------|----------|--------------|
| Sequential | 有，串行自动流转 | 支持 | 下标/命名属性 | 简单串行主干、分类输出头 |
| ModuleList | 无 | 不支持 | 数字下标 | 有序重复堆叠、同步多任务训练 |
| ModuleDict | 无 | 不支持 | 字符串语义key | 多任务分支、推理按需单支路 |

### 混合嵌套完整工程示例（Sequential + ModuleList + ModuleDict）
```python
class FullMixNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Sequential串行主干，自带数据流传递
        self.backbone = nn.Sequential(nn.Linear(10,64), nn.ReLU(), nn.Linear(64,64))
        # ModuleList堆叠3层特征变换块
        self.stack_blocks = nn.ModuleList([nn.Linear(64,64) for _ in range(3)])
        # ModuleDict多任务输出头
        self.task_heads = nn.ModuleDict({"cls":nn.Linear(64,10), "reg":nn.Linear(64,1)})

    def forward(self, x, task="cls"):
        x = self.backbone(x) # Sequential自动流转张量
        for blk in self.stack_blocks: # ModuleList手动循环堆叠
            x = blk(x)
        return self.task_heads[task](x) # ModuleDict按需选取任务分支
```

## 2.7 Module与容器高频避坑精简要点
1. `super().__init__()` 必须写，否则参数注册完全失效，无法训练、保存权重
2. 网络层必须赋值`self.xxx`，原生list/dict无法注册参数，梯度不更新
3. 模型推理调用 `model(x)`，禁止手动调用 `model.forward(x)`
4. ModuleList/ModuleDict不能单独传入张量推理，无内置数据流逻辑
5. Sequential自带串行计算逻辑，可独立作为子网络/主干
6. 选型准则：堆叠重复块用ModuleList、多分支按需推理用ModuleDict、简单串行主干用Sequential

# 三、torch.nn 各类网络层详解
## 3.1 线性层 nn.Linear（全连接层）
公式：$y = xW^T + b$
参数说明：
- in_features：输入特征维度
- out_features：输出特征维度
- bias：是否启用偏置项，默认 True

In [ ]:
# ====================== nn.Linear 全连接层使用演示 ======================
# 实例化线性层：输入5维特征，映射输出3维，自动迁移至对应设备
linear = nn.Linear(5, 3).to(device)
# 构造输入张量：batch_size=2条样本，每条样本5维特征
x = torch.randn(2, 5).to(device)
# 执行线性变换计算
output = linear(x)
# 打印输入、输出张量维度
print("输入shape:", x.shape)
print("输出shape:", output.shape)

## 3.2 激活函数独立层
nn包内置有状态层：nn.ReLU、nn.Sigmoid、nn.Tanh、nn.LeakyReLU、nn.Softmax、nn.LogSoftmax
配套无状态函数版本（推荐推理/临时计算）：F.relu()、F.sigmoid() 等

In [ ]:
# ====================== 主流激活函数数值对比演示 ======================
# 构造测试一维张量，包含负数、0、正数，直观展示各激活函数变换效果
x = torch.tensor([-2.0, -0.5, 0.0, 1.2]).to(device)
# 实例化三种最常用激活层
relu = nn.ReLU()     # ReLU：负值置0，正值保留
sigmoid = nn.Sigmoid() # Sigmoid：输出压缩至0~1区间，二分类概率
tanh = nn.Tanh()     # Tanh：输出压缩至-1~1区间，中心化特征

# 打印原始输入与各激活函数输出结果对比
print("原始x:", x)
print("ReLU:", relu(x))
print("Sigmoid:", sigmoid(x))
print("Tanh:", tanh(x))

## 3.3 卷积与池化层（CNN图像任务核心）
1. nn.Conv1d / Conv2d / Conv3d：一维/二维/三维卷积，图像通用Conv2d
2. nn.MaxPool2d / nn.AvgPool2d：最大池化、平均池化下采样
3. nn.Flatten：特征图展平，衔接卷积与全连接层
张量标准格式：Conv2d 输入 [batch, channel, height, width]

In [ ]:
# ====================== CNN卷积+池化+展平完整流程演示 ======================
# Conv2d参数说明：in_channels输入通道，out_channels输出通道，kernel_size卷积核尺寸，padding填充边缘保持尺寸不变
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1).to(device)
# MaxPool2d最大池化：kernel_size=2，stride=2，长宽各缩小一半，下采样降维
pool = nn.MaxPool2d(kernel_size=2, stride=2)
# Flatten展平层：将4维图像特征转为2维[batch, 特征总数]，送入全连接
flatten = nn.Flatten()

# 模拟图片输入张量：batch=1，3通道RGB，高32像素，宽32像素
img = torch.randn(1, 3, 32, 32).to(device)
# 第一步：卷积提取图像局部特征
c_out = conv(img)
# 第二步：池化下采样，压缩特征图尺寸，减少计算量
p_out = pool(c_out)
# 第三步：4维特征图展平为二维特征向量
f_out = flatten(p_out)

# 打印每一步张量维度变化，直观观察特征压缩过程
print("卷积后shape:", c_out.shape)
print("池化后shape:", p_out.shape)
print("展平后shape:", f_out.shape)

## 3.4 归一化层（加速收敛、稳定特征分布）
- nn.BatchNorm1d / BatchNorm2d：批量归一化，CNN/全连接通用
- nn.LayerNorm：层归一化，NLP Transformer主流
- InstanceNorm：实例归一化，风格迁移任务
核心作用：缓解梯度消失、大幅降低训练迭代轮数、提升泛化能力

## 3.5 正则化层（抑制过拟合）
- nn.Dropout：一维特征随机失活
- nn.Dropout2d：卷积特征图整体失活，CNN专用
关键特性：仅model.train()模式生效，model.eval()自动关闭，不影响推理精度

In [ ]:
# ====================== Dropout训练/推理模式行为对比 ======================
# 实例化Dropout层：p=0.3代表训练时随机30%神经元输出置0，防止过拟合
dropout = nn.Dropout(p=0.3).to(device)
# 构造全1测试张量，便于观察随机置0效果
x = torch.ones(10).to(device)
# 默认train模式，Dropout随机置零部分元素，每次运行结果不同
print("训练模式Dropout输出:", dropout(x))

# 切换至eval推理模式，Dropout完全失效，所有神经元输出保留，保证推理稳定
dropout.eval()
print("评估模式Dropout输出:", dropout(x))

# 四、损失函数 nn.*Loss（模型优化目标）
## 分类任务损失
- nn.CrossEntropyLoss：多分类专用，内部集成LogSoftmax，输入原始logits
- nn.BCELoss：二分类，输入0~1概率
- nn.BCEWithLogitsLoss：二分类，输入原始logits（推荐，数值更稳定）
- nn.NLLLoss：负对数似然，需手动前置LogSoftmax

## 回归任务损失
- nn.MSELoss：均方误差，连续值预测
- nn.L1Loss：平均绝对误差，抗异常值

In [ ]:
# ====================== CrossEntropyLoss 多分类损失演示 ======================
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== CrossEntropyLoss 3分类标准演示 =====================
# 任务：3分类，全局类别编号 0、1、2
# logits shape [batch_size=3, num_classes=3]
# 第1维3：一批3个样本；第2维3：固定总类别数=3
loss_fn = nn.CrossEntropyLoss()

# 3个样本，每个样本输出3个类别的原始预测得分logits
logits = torch.tensor([
    [1.2, 0.3, -0.5],   # 样本0：类别0、1、2预测分
    [0.1, 2.5, 0.4],    # 样本1：类别0、1、2预测分
    [-0.8, 0.2, 3.1]    # 样本2：类别0、1、2预测分
]).to(device)

# labels：3个样本各自的真实类别，覆盖全部0/1/2三类，直观体现3分类
labels = torch.tensor([0, 1, 2]).to(device)

# 内部Softmax可视化，直观看到每个样本输出3类概率
softmax = nn.Softmax(dim=1)
prob = softmax(logits)
print("各样本3类Softmax概率分布：\n", prob)

# 计算批次平均交叉熵损失
loss = loss_fn(logits, labels)
print(f"3分类批次平均损失值：{loss.item():.4f}")

# 五、优化器 torch.optim（参数梯度更新）
常用优化器适配场景：
- optim.SGD：随机梯度下降，配合momentum动量，传统CNN首选
- optim.Adam：自适应学习率动量，日常快速训练通用
- optim.AdamW：Adam改良，权重衰减分离，Transformer最优
- optim.RMSprop：RNN时序任务常用

核心超参：lr学习率、weight_decay权重衰减（L2正则）

In [ ]:
# ====================== 完整梯度更新五步标准流程 ======================
# 新建模型实例
model = Net(10, 32, 2).to(device)
# 初始化Adam优化器：传入模型全部可训练参数，lr学习率1e-3，weight_decay L2正则抑制过拟合
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 模拟单批次训练数据：8条10维特征样本
x = torch.randn(8, 10).to(device)
# 模拟真实二分类标签，取值0或1
y_true = torch.randint(0, 2, (8,)).to(device)

# 训练标准5步流程，顺序不可颠倒
# 1. zero_grad() 清空上一轮迭代累积梯度，梯度默认会累加
optimizer.zero_grad()
# 2. 前向传播：输入送入模型得到预测logits
y_pred = model(x)
# 3. 计算损失：对比预测与真实标签，得到损失标量
loss = nn.CrossEntropyLoss()(y_pred, y_true)
# 4. 反向传播：自动微分计算所有参数梯度，存入param.grad
loss.backward()
# 5. step()：优化器根据梯度、学习率更新模型所有权重参数
optimizer.step()

# 打印本次迭代损失数值
print(f"单次迭代损失: {loss.item():.4f}")

# 六、完整端到端训练+评估流水线实战
流程链路：模拟数据集构建 → 模型初始化 → 损失函数+优化器 → 多轮训练循环 → 每轮评估精度

In [ ]:
# ====================== 完整二分类训练评估流水线 ======================
# 1. 生成模拟二分类数据集
n_samples = 500 # 总样本数量500条
X = torch.randn(n_samples, 10).to(device) # 500行10列特征张量
# 构造标签规则：每条样本10维特征求和大于0则标签1，否则0，天然可分数据集
Y = (torch.sum(X, dim=1) > 0).long().to(device)

# 2. 模型、损失函数、优化器初始化
train_model = Net(10, 64, 2).to(device) # 训练模型实例
criterion = nn.CrossEntropyLoss() # 多分类损失函数
opt = optim.Adam(train_model.parameters(), lr=0.001) # Adam优化器

# 3. 完整多轮训练循环
epochs = 50 # 总训练轮次50轮
for epoch in range(epochs):
    # 切换模型至训练模式：Dropout、BatchNorm生效，开启梯度计算
    train_model.train()
    opt.zero_grad() # 清空本轮前梯度
    pred = train_model(X) # 全部样本前向推理
    loss = criterion(pred, Y) # 计算全量样本平均损失
    loss.backward() # 反向传播求梯度
    opt.step() # 更新权重

    # 每训练10轮执行一次模型评估，查看精度变化
    if (epoch + 1) % 10 == 0:
        train_model.eval() # 切换推理模式：关闭Dropout/BatchNorm随机行为
        # with torch.no_grad()上下文管理器：禁用梯度计算，节省显存、加速推理
        with torch.no_grad():
            eval_pred = train_model(X) # 全样本预测
            pred_label = torch.argmax(eval_pred, dim=1) # 取logits最大值下标作为预测类别
            acc = (pred_label == Y).float().mean() # 预测正确样本占比=准确率
        # 打印当前轮次损失与准确率
        print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

# 七、torch.nn API 速查表（对应菜鸟参考文档完整目录）
## 1. 基础模块容器
- nn.Module：所有网络基类
- nn.Sequential：串行流水线容器
- nn.ModuleList：有序层列表容器
- nn.ModuleDict：多分支字典容器
- nn.Parameter：手动定义可训练参数

## 2. 线性层 & 激活函数
Linear, Bilinear, ReLU, Sigmoid, Tanh, LeakyReLU, Softmax, LogSoftmax

## 3. 卷积/池化/图像操作层
Conv1d/2d/3d, MaxPool1d/2d/3d, AvgPool, Flatten, Upsample

## 4. 归一化 & 正则化层
BatchNorm1d/2d, LayerNorm, InstanceNorm, Dropout, Dropout2d

## 5. 损失函数合集
CrossEntropyLoss, BCEWithLogitsLoss, BCELoss, MSELoss, L1Loss, NLLLoss

## 6. NLP专用工具层
nn.Embedding 词嵌入层、nn.RNN / LSTM / GRU 循环神经网络

# 八、全局训练高频易错点汇总
## Module与容器踩坑
1. 忘记 super().__init__() → 参数不注册，权重无法更新、保存
2. 层存入原生list/dict而非三大专用容器 → 无梯度、无法迁移GPU
3. ModuleList/ModuleDict直接输入张量推理 → 无forward逻辑，代码报错

## 训练流程踩坑
1. 训练前忘记 optimizer.zero_grad() → 梯度累积，loss爆炸不收敛
2. 推理阶段未使用 model.eval() + torch.no_grad() → Dropout/BatchNorm干扰精度、占用显存
3. 模型与数据不在同一设备（CPU/GPU） → 张量设备不匹配报错

## 损失函数踩坑
1. CrossEntropyLoss 输入提前softmax → 重复对数运算，梯度异常
2. BCE损失输入未压缩至0~1区间 → loss数值溢出

## 容器选型踩坑
1. 多分支推理使用ModuleList数字下标 → 新增分支后全部下标失效
2. 重复堆叠网络使用Sequential → 代码大量重复，难以扩展层数